# Org Listings (Internal Marketplace)

Share data products within a Snowflake organization across accounts. Org listings replace legacy data exchanges for enterprise internal sharing.

**Key benefits:**
- Share curated data products with any account in your organization
- Consumers discover listings via the **Internal Marketplace** tab in Snowsight
- Governed by the provider — no data duplication, always up-to-date
- Supports metadata, sample queries, and usage instructions alongside the data

## 1. Setup — Provider Data

In [ ]:
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE DATABASE org_listing_demo;
CREATE SCHEMA org_listing_demo.shared;

USE ROLE SECURITYADMIN;
CREATE ROLE IF NOT EXISTS share_admin;
CREATE ROLE IF NOT EXISTS share_monitor;
GRANT ROLE share_admin TO ROLE SYSADMIN;
GRANT ROLE share_monitor TO ROLE share_admin;
GRANT CREATE SHARE ON ACCOUNT TO ROLE share_admin;
GRANT MANAGE SHARE TARGET ON ACCOUNT TO ROLE share_admin;
GRANT USAGE ON DATABASE org_listing_demo TO ROLE share_admin;
GRANT USAGE ON ALL SCHEMAS IN DATABASE org_listing_demo TO ROLE share_admin;
GRANT SELECT ON FUTURE TABLES IN SCHEMA org_listing_demo.shared TO ROLE share_admin;
GRANT CREATE VIEW ON SCHEMA org_listing_demo.shared TO ROLE share_admin;
GRANT IMPORTED PRIVILEGES ON DATABASE snowflake TO ROLE share_monitor;

USE ROLE ACCOUNTADMIN;
CREATE TABLE org_listing_demo.shared.product_catalog AS
SELECT p_partkey AS product_id, p_name AS product_name,
       p_brand AS brand, p_type AS product_type,
       p_retailprice AS retail_price
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.PART
LIMIT 5000;

GRANT SELECT ON ALL TABLES IN SCHEMA org_listing_demo.shared TO ROLE share_admin;

## 2. Create Secure Views

In [ ]:
USE ROLE share_admin;

CREATE OR REPLACE SECURE VIEW org_listing_demo.shared.v_product_catalog AS
SELECT product_id, product_name, brand, product_type, retail_price
FROM org_listing_demo.shared.product_catalog;

CREATE OR REPLACE SECURE VIEW org_listing_demo.shared.v_brand_summary AS
SELECT brand, COUNT(*) AS product_count, AVG(retail_price) AS avg_price
FROM org_listing_demo.shared.product_catalog
GROUP BY brand;

## 3. Create Share for Org Listing

In [ ]:
USE ROLE share_admin;

CREATE OR REPLACE SHARE org_product_catalog_share
    COMMENT = 'Product catalog for internal marketplace';
GRANT USAGE ON DATABASE org_listing_demo TO SHARE org_product_catalog_share;
GRANT USAGE ON SCHEMA org_listing_demo.shared TO SHARE org_product_catalog_share;
GRANT SELECT ON ALL VIEWS IN SCHEMA org_listing_demo.shared TO SHARE org_product_catalog_share;

DESCRIBE SHARE org_product_catalog_share;

## 4. Create Org Listing (Snowsight UI)

Steps to create an organization listing via Provider Studio:

1. Navigate to **Data Products** → **Provider Studio** → **+ Listing**
2. Select **"Only Specified Consumers"** for listing type
3. Under sharing scope, choose **"Accounts in Your Organization"**
4. Associate the share: `ORG_PRODUCT_CATALOG_SHARE`
5. Add metadata (title, description, sample queries)
6. Select target accounts within your organization
7. Publish

> All accounts in your organization will see this in the **Internal Marketplace** tab.
> Requires the Snowflake Organizations feature to be enabled.

## 5. Consumer Side — Install Org Listing

In [ ]:
-- Run in CONSUMER account after org listing is available:
-- Consumers see org listings in Data Products > Marketplace > Internal

-- After clicking "Get":
-- CREATE DATABASE product_catalog_db FROM SHARE <PROVIDER_ORG>.<PROVIDER_ACCT>.org_product_catalog_share;

-- Grant to local roles
-- GRANT IMPORTED PRIVILEGES ON DATABASE product_catalog_db TO ROLE analyst;

-- Verify
-- SELECT * FROM product_catalog_db.shared.v_product_catalog LIMIT 10;
-- SELECT * FROM product_catalog_db.shared.v_brand_summary;

## 6. Verify Share Status

In [ ]:
SHOW SHARES LIKE 'ORG_PRODUCT_CATALOG_SHARE';
SHOW GRANTS TO SHARE org_product_catalog_share;

## 7. Teardown

In [ ]:
USE ROLE ACCOUNTADMIN;
DROP SHARE IF EXISTS org_product_catalog_share;
DROP DATABASE IF EXISTS org_listing_demo;
DROP ROLE IF EXISTS share_admin;
DROP ROLE IF EXISTS share_monitor;

## References

- [Snowflake Listings Overview](https://docs.snowflake.com/en/user-guide/data-sharing-provider)
- [Creating a Listing](https://docs.snowflake.com/en/user-guide/data-sharing-provider-listings)
- [Working with Shares](https://docs.snowflake.com/en/user-guide/data-sharing-shares)
- [Snowflake Organizations](https://docs.snowflake.com/en/user-guide/organizations)